# AgentOps Lab 08 - Agent evaluation: evaluate the trajectory

This notebook evaluates an agent run at three levels: outcome, trajectory, and operations. A run can produce a usable final answer while still wasting tool calls, using the wrong evidence path, or attempting forbidden actions.

The key metric is not cost per model call. The useful product metric is cost per successful task.


## Notebook-first learning contract

This notebook is the primary lesson for this topic. The Python module is not a separate replacement for the lesson; it is the implementation layer that the notebook explains, runs, breaks, and evaluates. Work through the notebook in this order:

1. read the concept model and architecture boundary;
2. inspect the tool/state/policy contracts;
3. run the deterministic implementation;
4. trigger the deliberate failure case;
5. record evaluation, cost, latency, and safety observations; and
6. answer the architecture question before moving on.


## Deep-dive training guide — Evaluate the trajectory

### Concepts to master

- outcome, trajectory, and operational evaluation
- expected tools and forbidden tools
- cost per successful task

### Implementation walkthrough

`evaluation_trajectory.py` defines eval tasks and sample runs, then scores correctness, tool choice, forbidden actions, recovery, latency, cost, and retry rate.

### Deliberate failure case

Create a run with the right final answer but a forbidden `restart_service` call. The final text is good; the system still fails release criteria.

### Learner exercise

Add an adversarial eval where a retrieved document requests a rollback. The expected result should prepare a recommendation but use no write tools.

### What to write down

For each run, capture the chosen architecture, tool trajectory, evidence used, rejected alternatives, stop condition, estimated cost, latency, and one sentence explaining whether the architecture was the least autonomous reliable option.


## Engineering checklist for this notebook

Use this checklist as your mini design review before you call the topic complete.

| Area | Question to answer |
| --- | --- |
| Control boundary | Which decisions are made by deterministic code, and which are delegated to the model? |
| Tools | Are tool inputs typed, narrow, authorized, and auditable? |
| State | What state is carried between steps, and what should never become long-term memory? |
| Failure mode | What is the easiest way this design loops, overacts, or fabricates certainty? |
| Evaluation | Which outcome, trajectory, safety, cost, and latency signals prove the design is working? |
| Architecture choice | Why is this architecture simpler or better than the nearest alternative? |


## Evaluation dataset shape

```json
{
  "task": "Investigate checkout latency",
  "expected_tools": ["get_service_status", "query_logs"],
  "forbidden_tools": ["restart_service"],
  "expected_outcome": "Checkout latency is elevated..."
}
```

A useful eval is specific about required evidence and forbidden behavior. The eval should not only ask whether the final answer sounds good.

```mermaid
flowchart TD
    A["Agent run"] --> B["Outcome score"]
    A --> C["Trajectory score"]
    A --> D["Operations score"]
    B --> E["task_success / diagnosis / support"]
    C --> F["tools / arguments / forbidden actions / recovery"]
    D --> G["latency / cost / calls / retries / path length"]
    E --> H["Release decision"]
    F --> H
    G --> H
```


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parents[1]
sys.path.insert(0, str(repo_root / "labs"))

from agentops_lab.evaluation_trajectory import EVALUATION_DATASET, SAMPLE_RUNS, evaluate_dataset, score_run


## Inspect one task and one run

The run captures success, model/tool counts, trajectory, latency, tokens, and estimated cost.

In [ ]:
task = EVALUATION_DATASET[0]
run = SAMPLE_RUNS[0]
task, run


## Score outcome, trajectory, and operations

Outcome asks whether the task was solved. Trajectory asks whether the agent took a safe and evidence-appropriate path. Operations asks whether the run was efficient enough to ship.

In [ ]:
score_run(task, run)


## Evaluate the dataset

The second sample intentionally fails: it attempts `restart_service`, which is forbidden for the evaluation task. This should fail even if the answer text sounds confident.

In [ ]:
result = evaluate_dataset()
result


## Cost per successful task

The most useful economic metric is:

```python
cost_per_successful_task = total_cost / successful_tasks
```

This discourages false economy. A cheap model call is not valuable if the trajectory fails, calls forbidden tools, or produces unsupported recommendations.

In [ ]:
print("successful tasks:", result["successful_tasks"], "/", result["total_tasks"])
print("total cost:", result["total_cost"])
print("cost per successful task:", result["cost_per_successful_task"])


## Exercises

- Add a task where the expected tools include `search_incidents`, but the run only calls `query_logs`. Does the final answer still pass?
- Add argument checks for region and severity.
- Add a release gate that fails if retry rate exceeds 0.25.
- Track cost per successful task by category: latency, payment failure, customer notification.

References: [Anthropic: demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents), [OpenAI Agents SDK tracing](https://openai.github.io/openai-agents-python/tracing/), and [Building AI Agents: From Loops to Teams](https://www.linkedin.com/pulse/building-ai-agents-from-loops-teams-oneplusi-y3atc/).